# Comparison Tables for Paper

This notebook generates all tables comparing:
1. **True system** — the 20-reaction CRN being recovered
2. **LASSO (sparse-learning-CRN)** — fixed λ across T values; T=200 lambda sweep
3. **Bayesian spike-and-slab** — rate recovery and model selection

Each table is displayed inline and saved as a `.tex` file in this folder.

**L2 error** is computed in the 15-dimensional polynomial coefficient space for both methods,
making them directly comparable.

## 0. Setup

In [1]:
import sys, os
import numpy as np
import pandas as pd

# Add repo root to path (notebook lives in writing/, repo root is ..)
REPO_ROOT   = os.path.abspath('..')
sys.path.insert(0, REPO_ROOT)          # gives access to CRN_Simulation and src/
sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))  # direct import of parsing

SPARSE_RUNS  = os.path.join(REPO_ROOT, 'sparse_runs')
RESULTS_DIR  = os.path.join(REPO_ROOT, 'results')
DATA_DIR     = os.path.join(REPO_ROOT, 'data')
WRITING_DIR  = os.path.abspath('.')   # this notebook's folder = writing/

NETWORK_FILE  = os.path.join(DATA_DIR, 'example5_network.json')

# LASSO settings
FIXED_LAMBDA  = 0.01
SWEEP_LAMBDAS = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5]
T_VALUES      = [100, 200, 400]

def lam_str(lam): return str(lam).replace('.', 'p')
def sparse_run_dir(T, lam):
    return os.path.join(SPARSE_RUNS, f'example5_T{T}_lam{lam_str(lam)}')
def mcmc_result_dir(T):
    return os.path.join(RESULTS_DIR, f'example5_T{T}')

print('Repo root:', REPO_ROOT)


Repo root: /Users/suzanne/Documents/GitHub/BayesCRNInference


## 1. Load Network and Define Helpers

In [2]:
from src.parsing import load_reaction_network

result = load_reaction_network(NETWORK_FILE)
# Unpack return tuple
rxn_names_list   = result[2]   # list of 20 sampled reaction name strings
k_names_list     = result[3]   # list of 20 k-parameter names, e.g. 'k22'
compatible_rxns  = result[9]   # dict: tuple(stoich_change) -> [candidate reaction indices]
species_names    = result[10]  # ['A','B','X','Y']
reactant_mat     = result[7]   # (4, 210) reactant stoichiometry for all 210 reactions
param_vals       = result[5]   # {'k22': 1.046, ...} true rates for the 20 active reactions

sp   = species_names
n_sp = len(sp)

# True rate lookup: reaction_index -> rate
true_rate_by_idx = {int(k[1:]): v for k, v in param_vals.items()}

# Reaction name lookup: reaction_index -> human-readable string
name_by_idx = {int(k[1:]): rxn_names_list[i].rstrip(':') for i, k in enumerate(k_names_list)}

# Basis function labels (order matches sparse-learning output)
basis_labels = ['1'] + sp + [sp[i]+'*'+sp[j] for i in range(n_sp) for j in range(i, n_sp)]

# Display column order: 1 | A^2 A | B^2 B | X^2 X | Y^2 Y | AB AX AY BX BY XY
col_order = ['1','A*A','A','B*B','B','X*X','X','Y*Y','Y','A*B','A*X','A*Y','B*X','B*Y','X*Y']
col_idx   = [basis_labels.index(c) for c in col_order]

print('Species:', sp)
print('Basis labels:', basis_labels)
print(f'Active reactions ({len(true_rate_by_idx)}):', true_rate_by_idx)


Reaction network loaded from /Users/suzanne/Documents/GitHub/BayesCRNInference/data/example5_network.json
  Species:   ['A', 'B', 'X', 'Y']
  Reactions: 20 sampled  |  210 total in full CRN
  Unique stoichiometric changes: 130
Species: ['A', 'B', 'X', 'Y']
Basis labels: ['1', 'A', 'B', 'X', 'Y', 'A*A', 'A*B', 'A*X', 'A*Y', 'B*B', 'B*X', 'B*Y', 'X*X', 'X*Y', 'Y*Y']
Active reactions (20): {22: 1.046377380209165, 53: 0.7166996443671432, 155: 0.5897219661598223, 199: 0.7676002433735416, 63: 1.0673280680928032, 24: 0.2803159923659085, 208: 0.906127060897544, 163: 0.4221298458375011, 29: 2.203110446845896, 70: 0.452756813424444, 109: 1.2593443372879236, 87: 1.9789114814591777, 72: 1.5712998768102708, 204: 0.4886915332391283, 83: 1.0205988796694354, 5: 0.6781716941068556, 149: 0.4167214425597647, 164: 0.6456419355009646, 54: 1.2782894211721718, 18: 1.5168073835123705}


In [3]:
# ── Helper functions ──────────────────────────────────────────────────────────

def poly_vector_from_rates(change, rate_by_param_idx):
    """
    Build 15-dim polynomial coefficient vector for one stoichiometric channel.

    change             : list/tuple of 4 ints (stoichiometric change)
    rate_by_param_idx  : dict {param_index: rate} — estimated or true rates
                         Param_index i maps to compatible_rxns[tuple(change)][i]

    Uses the stochastic propensity expansion:
      unimolecular  A      ->  + rate * A
      bimolecular   A+B    ->  + rate * A*B
      bimolecular   2A     ->  + rate/2 * A^2  -  rate/2 * A
      zeroth-order  ∅      ->  + rate * 1
    """
    c    = np.zeros(len(basis_labels))
    key  = tuple(int(x) for x in change)
    if key not in compatible_rxns:
        return c
    candidates = compatible_rxns[key]
    for param_i, rate in rate_by_param_idx.items():
        if abs(rate) < 1e-12:
            continue
        rxn_idx = candidates[param_i]
        rv      = reactant_mat[:, rxn_idx]          # reactant stoich vector
        nz      = np.nonzero(rv)[0]
        if len(nz) == 0:
            c[basis_labels.index('1')] += rate
        elif len(nz) == 1:
            i, m = nz[0], int(rv[nz[0]])
            if m == 1:
                c[basis_labels.index(sp[i])] += rate
            elif m == 2:
                c[basis_labels.index(sp[i]+'*'+sp[i])] += rate / 2.0
                c[basis_labels.index(sp[i])]            -= rate / 2.0
        elif len(nz) == 2:
            c[basis_labels.index(sp[nz[0]]+'*'+sp[nz[1]])] += rate
    return c


def true_poly_vector(change):
    """True polynomial coefficient vector from known rates."""
    key = tuple(int(x) for x in change)
    if key not in compatible_rxns:
        return np.zeros(len(basis_labels))
    candidates = compatible_rxns[key]
    rate_by_param = {i: true_rate_by_idx.get(rxn, 0.0)
                     for i, rxn in enumerate(candidates)}
    return poly_vector_from_rates(change, rate_by_param)


def read_channel_info(run_dir):
    """Returns (changes, counts) from channel_info.txt."""
    path = os.path.join(run_dir, 'output', 'channel_info.txt')
    if not os.path.exists(path):
        return None, None
    with open(path) as f:
        lines = [l.strip() for l in f.read().strip().splitlines() if l.strip()]
    n_ch = int(lines[0].split()[0])
    changes = [list(map(int, lines[i+1].split())) for i in range(n_ch)]
    counts  = list(map(int, lines[n_ch+1].split()))
    return changes, counts


def load_omega(run_dir, ch_idx):
    """Load sparse-learning coefficient vector for channel ch_idx."""
    path = os.path.join(run_dir, 'output', f'omega_vec_for_channel_{ch_idx}.txt')
    if not os.path.exists(path):
        return None
    with open(path) as f:
        parts = f.read().strip().split()
    n = int(parts[0])
    return np.array([float(x) for x in parts[1:n+1]])


def lasso_poly_l2(run_dir, changes, ref_changes=None):
    """
    Compute polynomial-space L2 error for each channel in run_dir.
    ref_changes : channel list from a reference run (e.g. T=100).
                  If provided, returns one L2 per ref_changes entry (NaN if missing).
    Returns dict: change_tuple -> L2
    """
    l2_by_change = {}
    for ch_idx, change in enumerate(changes):
        omega = load_omega(run_dir, ch_idx)
        if omega is None:
            continue
        # omega is in basis_labels order; reorder to col_order for consistency
        true_c = true_poly_vector(change)
        l2     = np.linalg.norm(omega - true_c)
        l2_by_change[tuple(change)] = l2
    return l2_by_change


def load_mcmc_summary(T):
    """Load mcmc_summary.xlsx for a given T. Returns DataFrame or None."""
    path = os.path.join(mcmc_result_dir(T), 'mcmc_summary.xlsx')
    if not os.path.exists(path):
        return None
    return pd.read_excel(path)


def mcmc_poly_l2(df_mcmc, changes):
    """
    Compute polynomial-space L2 error for each channel from MCMC summary.
    Returns dict: change_tuple -> L2
    """
    # Uses count_to_change_ref (built from T=100 channel info; channels are same across T)
    l2_by_change = {}
    for run_idx, grp in df_mcmc.groupby('Run_Index'):
        cnt    = grp['Count'].iloc[0]
        # find stoich change by matching count to the reference channel list
        change = count_to_change_ref.get(cnt)
        if change is None:
            continue
        key = tuple(change)
        # build rate_by_param_idx from posterior means
        rate_by_param = {int(row['Param_Index']): row['Mean']
                         for _, row in grp.iterrows()}
        est_c  = poly_vector_from_rates(change, rate_by_param)
        true_c = true_poly_vector(change)
        l2_by_change[key] = np.linalg.norm(est_c - true_c)
    return l2_by_change


print('Helpers defined.')

Helpers defined.


In [5]:
# Load reference channel info (T=100 — establishes channel order and obs counts)
ref_dir = sparse_run_dir(100, FIXED_LAMBDA)
ref_changes, ref_counts = read_channel_info(ref_dir)
assert ref_changes is not None, f'Cannot find channel_info in {ref_dir}'

count_to_change_ref = {cnt: ch for ch, cnt in zip(ref_changes, ref_counts)}

print(f'Reference channels ({len(ref_changes)}):')
for ch, cnt in zip(ref_changes, ref_counts):
    print(f'  {ch}  obs={cnt}')

Reference channels (15):
  [-2, 0, 0, 0]  obs=60
  [-2, 0, 1, 1]  obs=129
  [-2, 1, 0, 0]  obs=230
  [0, -2, 1, 0]  obs=615
  [0, 0, -1, 1]  obs=138
  [0, 0, 0, -1]  obs=310
  [0, 0, 0, 1]  obs=184
  [0, 0, 1, -1]  obs=136
  [0, 1, -1, 0]  obs=427
  [0, 1, -1, 1]  obs=317
  [0, 1, 0, -1]  obs=326
  [0, 1, 0, 0]  obs=378
  [0, 2, 0, 0]  obs=64
  [1, -1, 0, 0]  obs=580
  [1, 0, 0, 0]  obs=255


## Table 1 — True System Description

The 20 active reactions that make up the true network, grouped by stoichiometric channel.
The full candidate CRN contains 210 possible reactions over 4 species with polynomial order 2.

In [6]:
def fmt_reaction(name_str):
    """Convert 'A_to_A+B' style string to 'A → A+B' style."""
    s = name_str.replace('Empty', '∅').replace('2A','2A').replace('_to_',' → ')
    return s

def fmt_change(change):
    parts = []
    for s, v in zip(sp, change):
        if v != 0:
            sgn = '+' if v > 0 else ''
            parts.append(f'{sgn}{v}{s}')
    return ', '.join(parts) if parts else '0'

# Build table rows
rows = []
for ch_idx, (change, obs) in enumerate(zip(ref_changes, ref_counts)):
    key = tuple(change)
    candidates = compatible_rxns.get(key, [])
    active = [(i, rxn_idx) for i, rxn_idx in enumerate(candidates)
              if true_rate_by_idx.get(rxn_idx, 0) > 0]
    for param_i, rxn_idx in active:
        rate = true_rate_by_idx[rxn_idx]
        name = fmt_reaction(name_by_idx.get(rxn_idx, f'rxn_{rxn_idx}'))
        rows.append({
            'Ch': ch_idx,
            'Change': fmt_change(change),
            'Reaction': name,
            'k (true)': f'{rate:.4f}',
            'Obs (T=100)': obs,
        })

df_true = pd.DataFrame(rows)
# Suppress repeated Ch/Change/Obs for multi-reaction channels
df_display = df_true.copy()
for col in ['Ch','Change','Obs (T=100)']:
    df_display[col] = df_display[col].where(~df_display.duplicated(subset=['Ch']), other='')

print(df_display.to_string(index=False))

Ch        Change  Reaction k (true) Obs (T=100)
 0           -2A    2A → ∅   0.4528          60
 1 -2A, +1X, +1Y  2A → X+Y   1.0206         129
 2      -2A, +1B    2A → B   1.5713         230
 3      -2B, +1X    2B → X   1.9789         615
 4      -1X, +1Y  X+Y → 2Y   0.4887         138
 5           -1Y   A+Y → A   0.5897         310
             -1Y   X+Y → X   0.7676         310
 6           +1Y   A → A+Y   0.2803         184
                    Y → 2Y   1.0673            
 7      +1X, -1Y A+Y → A+X   0.6456         136
 8      +1B, -1X  2X → B+X   1.2593         427
                 A+X → A+B   0.4167            
 9 +1B, -1X, +1Y   X → B+Y   1.2783         317
10      +1B, -1Y A+Y → A+B   0.4221         326
                 X+Y → B+X   0.9061            
11           +1B   A → A+B   1.0464         378
                   X → B+X   0.7167            
12           +2B    ∅ → 2B   0.6782          64
13      +1A, -1B     B → A   2.2031         580
14           +1A    A → 2A   1.5168     

In [8]:
# Save Table 1 as LaTeX — clean version
def fmt_rxn_latex(name_str):
    """Convert 'A_to_A+B' to LaTeX math string."""
    s = name_str.replace('Empty', r'\emptyset')
    left, right = s.split('_to_')
    return f'${left} \\to {right}$'

def fmt_change_latex(change):
    parts = []
    for s, v in zip(sp, change):
        if v != 0:
            sgn = '+' if v > 0 else ''
            parts.append(f'${sgn}{v}{s}$')
    return ', '.join(parts) if parts else '$0$'

lines = []
lines.append(r'% Table 1 — True CRN system description')
lines.append(r'\begin{table}[htbp]')
lines.append(r'\centering')
lines.append(r'\caption{%')
lines.append(r'  The true 20-reaction CRN over species $A,B,X,Y$.')
lines.append(r'  Channels group reactions sharing the same net stoichiometric change.')
lines.append(r'  Both the sparse-learning and Bayesian methods infer one independent')
lines.append(r'  propensity per channel; within each channel up to 5 bimolecular')
lines.append(r'  mass-action reactions are candidates.')
lines.append(r'  Obs\ $=$ observed firings in the $T{=}100$ trajectory set.')
lines.append(r'}')
lines.append(r'\label{tab:true-system}')
lines.append(r'\begin{tabular}{rlllr}')
lines.append(r'\toprule')
lines.append(r'\textbf{Ch} & \textbf{Change} & \textbf{Reaction} & \textbf{Rate $k$} & \textbf{Obs} \\')
lines.append(r'\midrule')

prev_ch = None
for _, row in df_true.iterrows():
    ch_i   = row['Ch']
    ch     = ref_changes[ch_i]
    k_val  = float(row['k (true)'])
    obs    = row['Obs (T=100)']
    # Find the reaction index that has this rate
    candidates = compatible_rxns[tuple(ch)]
    rxn_idx = next((r for r in candidates if abs(true_rate_by_idx.get(r, 0) - k_val) < 1e-4), None)
    rxn_lat = fmt_rxn_latex(name_by_idx[rxn_idx]) if rxn_idx and rxn_idx in name_by_idx else '---'
    ch_str  = str(ch_i) if ch_i != prev_ch else ''
    chg_str = fmt_change_latex(ch) if ch_i != prev_ch else ''
    obs_str = str(obs) if ch_i != prev_ch else ''
    lines.append(f'{ch_str} & {chg_str} & {rxn_lat} & {k_val:.4f} & {obs_str} \\\\')
    prev_ch = ch_i

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(r'\end{table}')

tex1 = '\n'.join(lines)
out1 = os.path.join(WRITING_DIR, 'table1_true_system.tex')
with open(out1, 'w') as f:
    f.write(tex1)
print(f'Saved: {out1}')
print(tex1)


Saved: /Users/suzanne/Documents/GitHub/BayesCRNInference/writing/table1_true_system.tex
% Table 1 — True CRN system description
\begin{table}[htbp]
\centering
\caption{%
  The true 20-reaction CRN over species $A,B,X,Y$.
  Channels group reactions sharing the same net stoichiometric change.
  Both the sparse-learning and Bayesian methods infer one independent
  propensity per channel; within each channel up to 5 bimolecular
  mass-action reactions are candidates.
  Obs\ $=$ observed firings in the $T{=}100$ trajectory set.
}
\label{tab:true-system}
\begin{tabular}{rlllr}
\toprule
\textbf{Ch} & \textbf{Change} & \textbf{Reaction} & \textbf{Rate $k$} & \textbf{Obs} \\
\midrule
0 & $-2A$ & $2A \to \emptyset$ & 0.4528 & 60 \\
1 & $-2A$, $+1X$, $+1Y$ & $2A \to X+Y$ & 1.0206 & 129 \\
2 & $-2A$, $+1B$ & $2A \to B$ & 1.5713 & 230 \\
3 & $-2B$, $+1X$ & $2B \to X$ & 1.9789 & 615 \\
4 & $-1X$, $+1Y$ & $X+Y \to 2Y$ & 0.4887 & 138 \\
5 & $-1Y$ & $A+Y \to A$ & 0.5897 & 310 \\
 &  & $X+Y \to X$ & 0.7

## Table 2a — LASSO Fixed λ=0.01, All T Values

Polynomial-space L² error per channel for T=100, 200, 400.

In [9]:
# Compute L2 for each T
lasso_l2_fixed = {}   # T -> {change_tuple -> L2}
for T in T_VALUES:
    rdir = sparse_run_dir(T, FIXED_LAMBDA)
    ch, _ = read_channel_info(rdir)
    if ch is None:
        print(f'T={T}: results not found — skipping')
        lasso_l2_fixed[T] = {}
        continue
    lasso_l2_fixed[T] = lasso_poly_l2(rdir, ch)
    print(f'T={T}: {len(lasso_l2_fixed[T])} channels loaded')

# Build display DataFrame
rows2a = []
for ch, cnt in zip(ref_changes, ref_counts):
    key = tuple(ch)
    row = {'Ch': ref_changes.index(ch), 'Change': fmt_change(ch), 'Obs': cnt}
    for T in T_VALUES:
        l2 = lasso_l2_fixed[T].get(key, float('nan'))
        row[f'T={T}'] = f'{l2:.3f}' if not np.isnan(l2) else '—'
    rows2a.append(row)

df_2a = pd.DataFrame(rows2a)
df_2a.index = df_2a['Ch']
df_2a = df_2a.drop(columns='Ch')
print(df_2a.to_string())

T=100: 15 channels loaded
T=200: 15 channels loaded
T=400: 15 channels loaded
           Change  Obs  T=100  T=200  T=400
Ch                                         
0             -2A   60  0.487  0.374  0.317
1   -2A, +1X, +1Y  129  0.754  0.851  0.993
2        -2A, +1B  230  0.962  0.879  0.938
3        -2B, +1X  615  0.843  1.005  1.009
4        -1X, +1Y  138  0.226  0.292  0.320
5             -1Y  310  0.632  0.411  0.174
6             +1Y  184  0.484  0.451  0.178
7        +1X, -1Y  136  0.143  0.188  0.168
8        +1B, -1X  427  0.832  0.816  0.686
9   +1B, -1X, +1Y  317  0.360  0.308  0.182
10       +1B, -1Y  326  0.209  0.160  0.153
11            +1B  378  0.713  0.342  0.283
12            +2B   64  0.797  0.163  0.519
13       +1A, -1B  580  0.991  0.411  0.195
14            +1A  255  0.408  0.262  0.152


In [10]:
# Save Table 2a as LaTeX
def latex_2a(df, T_values, lam):
    lines = []
    lines.append(r'% Table 2a — LASSO polynomial L2 error, fixed lambda')
    lines.append(r'\begin{table}[htbp]')
    lines.append(r'\centering')
    lines.append(r'\caption{%')
    lines.append(rf'  Polynomial-space $L^2$ error for sparse-learning-CRN (fixed $\lambda={lam}$)')
    lines.append(r'  across trajectory lengths.')
    lines.append(r'  $L^2 = \|\hat{{c}} - c_{{\mathrm{{true}}}}\|_2$ over the 15-dimensional')
    lines.append(r'  polynomial basis. Obs $=$ observed firings in $T{=}100$ trajectories.')
    lines.append(r'}')
    lines.append(r'\label{tab:lasso-fixed-lambda}')
    col_spec = 'rllr' + 'r'*len(T_values)
    lines.append(rf'\begin{{tabular}}{{{col_spec}}}')
    lines.append(r'\toprule')
    t_headers = ' & '.join([f'$T={T}$' for T in T_values])
    lines.append(rf'\textbf{{Ch}} & \textbf{{Change}} & \textbf{{Obs}} & {t_headers} \\')
    lines.append(r'\midrule')
    for ch_i, (ch, cnt) in enumerate(zip(ref_changes, ref_counts)):
        key  = tuple(ch)
        chg  = fmt_change_latex(ch)
        vals = []
        for T in T_values:
            l2 = lasso_l2_fixed[T].get(key, float('nan'))
            vals.append(f'{l2:.3f}' if not np.isnan(l2) else '---')
        lines.append(f'{ch_i} & {chg} & {cnt} & {" & ".join(vals)} \\\\')
    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')
    return '\n'.join(lines)

tex2a = latex_2a(df_2a, T_VALUES, FIXED_LAMBDA)
out2a = os.path.join(WRITING_DIR, 'table2a_lasso_fixed_lambda.tex')
with open(out2a, 'w') as f:
    f.write(tex2a)
print(f'Saved: {out2a}')
print(tex2a)

Saved: /Users/suzanne/Documents/GitHub/BayesCRNInference/writing/table2a_lasso_fixed_lambda.tex
% Table 2a — LASSO polynomial L2 error, fixed lambda
\begin{table}[htbp]
\centering
\caption{%
  Polynomial-space $L^2$ error for sparse-learning-CRN (fixed $\lambda=0.01$)
  across trajectory lengths.
  $L^2 = \|\hat{{c}} - c_{{\mathrm{{true}}}}\|_2$ over the 15-dimensional
  polynomial basis. Obs $=$ observed firings in $T{=}100$ trajectories.
}
\label{tab:lasso-fixed-lambda}
\begin{tabular}{rllrrrr}
\toprule
\textbf{Ch} & \textbf{Change} & \textbf{Obs} & $T=100$ & $T=200$ & $T=400$ \\
\midrule
0 & $-2A$ & 60 & 0.487 & 0.374 & 0.317 \\
1 & $-2A$, $+1X$, $+1Y$ & 129 & 0.754 & 0.851 & 0.993 \\
2 & $-2A$, $+1B$ & 230 & 0.962 & 0.879 & 0.938 \\
3 & $-2B$, $+1X$ & 615 & 0.843 & 1.005 & 1.009 \\
4 & $-1X$, $+1Y$ & 138 & 0.226 & 0.292 & 0.320 \\
5 & $-1Y$ & 310 & 0.632 & 0.411 & 0.174 \\
6 & $+1Y$ & 184 & 0.484 & 0.451 & 0.178 \\
7 & $+1X$, $-1Y$ & 136 & 0.143 & 0.188 & 0.168 \\
8 & $+1B$, $-1X$ 

## Table 2b — LASSO T=200 Lambda Sweep

Polynomial L² error across λ values at fixed T=200.

In [11]:
# Compute L2 for each lambda at T=200
lasso_l2_sweep = {}  # lam -> {change_tuple -> L2}
for lam in SWEEP_LAMBDAS:
    rdir = sparse_run_dir(200, lam)
    ch, _ = read_channel_info(rdir)
    if ch is None:
        print(f'λ={lam}: results not found — skipping')
        lasso_l2_sweep[lam] = {}
        continue
    lasso_l2_sweep[lam] = lasso_poly_l2(rdir, ch)
    print(f'λ={lam}: {len(lasso_l2_sweep[lam])} channels loaded')

# Observation counts for T=200
ref_dir_200 = sparse_run_dir(200, FIXED_LAMBDA)
_, ref_counts_200 = read_channel_info(ref_dir_200)

rows2b = []
for ch, cnt100, cnt200 in zip(ref_changes, ref_counts,
                               ref_counts_200 if ref_counts_200 else ref_counts):
    key = tuple(ch)
    row = {'Ch': ref_changes.index(ch), 'Change': fmt_change(ch), 'Obs': cnt200}
    for lam in SWEEP_LAMBDAS:
        l2 = lasso_l2_sweep[lam].get(key, float('nan'))
        # Bold minimum
        row[f'λ={lam}'] = l2
    rows2b.append(row)

df_2b = pd.DataFrame(rows2b).set_index('Ch')

# Display with formatting
df_2b_display = df_2b.copy()
for col in [f'λ={lam}' for lam in SWEEP_LAMBDAS]:
    df_2b_display[col] = df_2b_display[col].apply(
        lambda v: f'{v:.3f}' if not np.isnan(v) else '—')
print(df_2b_display.to_string())

λ=0.001: 15 channels loaded
λ=0.005: 15 channels loaded
λ=0.01: 15 channels loaded
λ=0.05: 15 channels loaded
λ=0.1: 15 channels loaded
λ=0.5: 15 channels loaded
           Change   Obs λ=0.001 λ=0.005 λ=0.01 λ=0.05  λ=0.1  λ=0.5
Ch                                                                 
0             -2A   115   1.087   0.431  0.374  0.294  0.251  0.245
1   -2A, +1X, +1Y   263   1.477   0.919  0.851  0.632  0.570  0.556
2        -2A, +1B   452   0.919   0.881  0.879  0.866  0.832  0.833
3        -2B, +1X  1185   1.215   1.044  1.005  0.693  0.953  1.032
4        -1X, +1Y   252   1.867   0.763  0.292  0.083  0.029  0.073
5             -1Y   626   0.750   0.575  0.411  0.078  0.057  0.201
6             +1Y   398   0.540   0.378  0.451  0.481  0.648  1.124
7        +1X, -1Y   264   0.388   0.234  0.188  0.078  0.038  0.120
8        +1B, -1X   860   1.153   0.993  0.816  0.695  0.668  0.690
9   +1B, -1X, +1Y   603   0.594   0.465  0.308  0.155  0.322  1.315
10       +1B, -1Y   62

In [12]:
# Save Table 2b as LaTeX — bold the minimum L2 per row
lines = []
lines.append(r'% Table 2b — LASSO lambda sweep at T=200')
lines.append(r'\begin{table}[htbp]')
lines.append(r'\centering')
lines.append(r'\caption{%')
lines.append(r'  Polynomial-space $L^2$ error for sparse-learning-CRN at $T{=}200$')
lines.append(r'  across regularisation strengths $\lambda$.')
lines.append(r'  \textbf{Bold}: minimum $L^2$ per channel.')
lines.append(r'  No single $\lambda$ minimises error across all channels simultaneously,')
lines.append(r'  illustrating the fundamental limitation of global regularisation.')
lines.append(r'}')
lines.append(r'\label{tab:lasso-lambda-sweep}')
n_lam = len(SWEEP_LAMBDAS)
lines.append(r'\begin{tabular}{rlr' + 'r'*n_lam + r'}')
lines.append(r'\toprule')
lam_headers = ' & '.join([f'$\\lambda={lam}$' for lam in SWEEP_LAMBDAS])
lines.append(rf'\textbf{{Ch}} & \textbf{{Change}} & \textbf{{Obs}} & {lam_headers} \\')
lines.append(r'\midrule')

for ch_i, (ch, cnt200) in enumerate(zip(ref_changes,
                                         ref_counts_200 if ref_counts_200 else ref_counts)):
    key = tuple(ch)
    l2s = [lasso_l2_sweep[lam].get(key, float('nan')) for lam in SWEEP_LAMBDAS]
    min_l2 = min((v for v in l2s if not np.isnan(v)), default=float('nan'))
    cells = []
    for v in l2s:
        s = f'{v:.3f}' if not np.isnan(v) else '---'
        if not np.isnan(v) and abs(v - min_l2) < 1e-6:
            s = r'\textbf{' + s + r'}'
        cells.append(s)
    chg = fmt_change_latex(ch)
    lines.append(f'{ch_i} & {chg} & {cnt200} & {" & ".join(cells)} \\\\')

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(r'\end{table}')

tex2b = '\n'.join(lines)
out2b = os.path.join(WRITING_DIR, 'table2b_lasso_lambda_sweep.tex')
with open(out2b, 'w') as f:
    f.write(tex2b)
print(f'Saved: {out2b}')
print(tex2b)

Saved: /Users/suzanne/Documents/GitHub/BayesCRNInference/writing/table2b_lasso_lambda_sweep.tex
% Table 2b — LASSO lambda sweep at T=200
\begin{table}[htbp]
\centering
\caption{%
  Polynomial-space $L^2$ error for sparse-learning-CRN at $T{=}200$
  across regularisation strengths $\lambda$.
  \textbf{Bold}: minimum $L^2$ per channel.
  No single $\lambda$ minimises error across all channels simultaneously,
  illustrating the fundamental limitation of global regularisation.
}
\label{tab:lasso-lambda-sweep}
\begin{tabular}{rlrrrrrrr}
\toprule
\textbf{Ch} & \textbf{Change} & \textbf{Obs} & $\lambda=0.001$ & $\lambda=0.005$ & $\lambda=0.01$ & $\lambda=0.05$ & $\lambda=0.1$ & $\lambda=0.5$ \\
\midrule
0 & $-2A$ & 115 & 1.087 & 0.431 & 0.374 & 0.294 & 0.251 & \textbf{0.245} \\
1 & $-2A$, $+1X$, $+1Y$ & 263 & 1.477 & 0.919 & 0.851 & 0.632 & 0.570 & \textbf{0.556} \\
2 & $-2A$, $+1B$ & 452 & 0.919 & 0.881 & 0.879 & 0.866 & \textbf{0.832} & 0.833 \\
3 & $-2B$, $+1X$ & 1185 & 1.215 & 1.044 & 1.0

## Table 3 — Bayesian Spike-and-Slab Results

For each channel and each T value (loaded if available): posterior mean rates, Prob(off),
95% credible intervals, and **polynomial-space L² error** (same metric as Tables 2a/2b).

In [13]:
# Load MCMC summaries for all available T values
mcmc_dfs = {}
for T in T_VALUES:
    df = load_mcmc_summary(T)
    if df is not None:
        mcmc_dfs[T] = df
        print(f'T={T}: loaded {len(df)} parameter rows')
    else:
        print(f'T={T}: mcmc_summary.xlsx not found — will be marked as pending')

# Compute polynomial L2 for each available T
mcmc_l2 = {}  # T -> {change_tuple -> L2}
for T, df in mcmc_dfs.items():
    mcmc_l2[T] = mcmc_poly_l2(df, ref_changes)
    print(f'T={T}: computed L2 for {len(mcmc_l2[T])} channels')

T=100: loaded 51 parameter rows
T=200: mcmc_summary.xlsx not found — will be marked as pending
T=400: mcmc_summary.xlsx not found — will be marked as pending
T=100: computed L2 for 15 channels


In [14]:
# ── Summary table: one row per channel, L2 columns for each T ──────────────

rows3 = []
for ch_i, (ch, cnt) in enumerate(zip(ref_changes, ref_counts)):
    key = tuple(ch)
    row = {'Ch': ch_i, 'Change': fmt_change(ch), 'Obs (T=100)': cnt}
    for T in T_VALUES:
        l2 = mcmc_l2.get(T, {}).get(key, float('nan'))
        row[f'Bayes L2 T={T}'] = f'{l2:.3f}' if not np.isnan(l2) else '(running)'
    # Also add LASSO L2 at fixed lambda for direct comparison
    for T in T_VALUES:
        l2 = lasso_l2_fixed.get(T, {}).get(key, float('nan'))
        row[f'LASSO L2 T={T}'] = f'{l2:.3f}' if not np.isnan(l2) else '—'
    rows3.append(row)

df_3 = pd.DataFrame(rows3).set_index('Ch')

# Interleave columns: for each T, show Bayes then LASSO
cols_ordered = ['Change', 'Obs (T=100)']
for T in T_VALUES:
    cols_ordered += [f'Bayes L2 T={T}', f'LASSO L2 T={T}']
print(df_3[cols_ordered].to_string())

           Change  Obs (T=100) Bayes L2 T=100 LASSO L2 T=100 Bayes L2 T=200 LASSO L2 T=200 Bayes L2 T=400 LASSO L2 T=400
Ch                                                                                                                      
0             -2A           60          0.008          0.487      (running)          0.374      (running)          0.317
1   -2A, +1X, +1Y          129          0.060          0.754      (running)          0.851      (running)          0.993
2        -2A, +1B          230          0.061          0.962      (running)          0.879      (running)          0.938
3        -2B, +1X          615          0.056          0.843      (running)          1.005      (running)          1.009
4        -1X, +1Y          138          0.045          0.226      (running)          0.292      (running)          0.320
5             -1Y          310          0.108          0.632      (running)          0.411      (running)          0.174
6             +1Y          184  

In [15]:
# ── Detailed rate table for T=100 (and other T if available) ─────────────────

def show_mcmc_detail(T):
    """Print per-channel, per-reaction detail for MCMC at a given T."""
    if T not in mcmc_dfs:
        print(f'T={T} not available yet.')
        return
    df = mcmc_dfs[T]

    rows = []
    for run_idx, grp in df.groupby('Run_Index'):
        cnt    = grp['Count'].iloc[0]
        change = count_to_change_ref.get(cnt)
        if change is None:
            continue
        key       = tuple(change)
        candidates = compatible_rxns.get(key, [])
        poly_l2   = mcmc_l2.get(T, {}).get(key, float('nan'))

        # All params for this run
        for _, r in grp.sort_values('Param_Index').iterrows():
            pi       = int(r['Param_Index'])
            rxn_idx  = candidates[pi] if pi < len(candidates) else None
            rxn_name = name_by_idx.get(rxn_idx, '—') if rxn_idx else '—'
            rows.append({
                'Ch':       ref_changes.index(list(change)),
                'Change':   fmt_change(change),
                'Reaction': fmt_reaction(rxn_name),
                'True k':   f"{r['True_Theta']:.4f}",
                'Post mean': f"{r['Mean']:.4f}",
                'Post std':  f"{r['Std']:.4f}",
                '95% CI':   f"[{r['CI_lower']:.3f}, {r['CI_upper']:.3f}]",
                'Prob Off':  f"{r['Prob_Off']:.4f}",
                'Poly L2':  f'{poly_l2:.4f}' if not np.isnan(poly_l2) else '—',
            })

    df_detail = pd.DataFrame(rows)
    # Suppress repeated Ch/Change/Poly L2 within each channel
    for col in ['Ch','Change','Poly L2']:
        df_detail[col] = df_detail[col].where(~df_detail.duplicated(subset=['Ch']), other='')

    print(f'\n=== Bayesian Spike-and-Slab Detail — T={T} ===')
    print(df_detail.to_string(index=False))
    return df_detail

for T in T_VALUES:
    show_mcmc_detail(T)


=== Bayesian Spike-and-Slab Detail — T=100 ===
Ch        Change  Reaction True k Post mean Post std         95% CI Prob Off Poly L2
14           +1A         — 0.0000    0.0000   0.0000 [0.000, 0.000]   1.0000  0.0405
             +1A    A → 2A 1.5168    1.4763   0.0921 [1.302, 1.661]   0.0000  0.0405
                         — 0.0000    0.0000   0.0000 [0.000, 0.000]   1.0000        
                         — 0.0000    0.0000   0.0000 [0.000, 0.000]   1.0000        
                         — 0.0000    0.0000   0.0000 [0.000, 0.000]   1.0000        
11           +1B         — 0.0000    0.0003   0.0117 [0.000, 0.000]   0.9987  0.1685
                   A → A+B 1.0464    1.2137   0.1350 [0.948, 1.488]   0.0000        
                         — 0.0000    0.0000   0.0000 [0.000, 0.000]   1.0000        
                   X → B+X 0.7167    0.7369   0.0970 [0.553, 0.937]   0.0000        
                         — 0.0000    0.0000   0.0000 [0.000, 0.000]   1.0000        
 6           +1Y 

In [16]:
# Save Table 3 (L2 comparison) as LaTeX
lines = []
lines.append(r'% Table 3 — Bayesian vs LASSO polynomial L2 comparison')
lines.append(r'\begin{table}[htbp]')
lines.append(r'\centering')
lines.append(r'\small')
lines.append(r'\caption{%')
lines.append(r'  Polynomial-space $L^2$ error comparison:')
lines.append(r'  Bayesian spike-and-slab (\textbf{Bayes}) vs.\ sparse-learning-CRN')
lines.append(r'  (\textbf{LASSO}, fixed $\lambda = ' + str(FIXED_LAMBDA) + r'$).')
lines.append(r'  $L^2 = \|\hat{c} - c_{\mathrm{true}}\|_2$ in the 15-dimensional')
lines.append(r'  polynomial basis. ``(r)\rq\rq\/ = results still running.')
lines.append(r'}')
lines.append(r'\label{tab:comparison-l2}')

# Column spec: Ch Change Obs | (Bayes LASSO) x 3
lines.append(r'\begin{tabular}{rlr|rr|rr|rr}')
lines.append(r'\toprule')
# Header row 1 — T groupings
t_groups = ' & '.join([rf'\multicolumn{{2}}{{c}}{{$T={T}$}}' for T in T_VALUES])
lines.append(rf'\multicolumn{{3}}{{c}}{{}} & {t_groups} \\')
for T in T_VALUES:
    lines[-1] = lines[-1]  # already set
cmidrules = ''.join([rf'\cmidrule(lr){{{4+2*i}-{5+2*i}}}' for i in range(len(T_VALUES))])
lines.append(cmidrules)
# Header row 2 — column names
sub_headers = ' & '.join(['Bayes & LASSO'] * len(T_VALUES))
lines.append(rf'\textbf{{Ch}} & \textbf{{Change}} & \textbf{{Obs}} & {sub_headers} \\')
lines.append(r'\midrule')

for ch_i, (ch, cnt) in enumerate(zip(ref_changes, ref_counts)):
    key = tuple(ch)
    chg = fmt_change_latex(ch)
    cells = []
    for T in T_VALUES:
        bl2 = mcmc_l2.get(T, {}).get(key, float('nan'))
        ll2 = lasso_l2_fixed.get(T, {}).get(key, float('nan'))
        bs = f'{bl2:.3f}' if not np.isnan(bl2) else r'\textit{(r)}'
        ls = f'{ll2:.3f}' if not np.isnan(ll2) else '---'
        # Bold whichever is smaller (if both available)
        if not np.isnan(bl2) and not np.isnan(ll2):
            if bl2 <= ll2:
                bs = r'\textbf{' + bs + r'}'
            else:
                ls = r'\textbf{' + ls + r'}'
        cells.append(f'{bs} & {ls}')
    lines.append(f'{ch_i} & {chg} & {cnt} & {" & ".join(cells)} \\\\')

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(r'\end{table}')

tex3 = '\n'.join(lines)
out3 = os.path.join(WRITING_DIR, 'table3_comparison_l2.tex')
with open(out3, 'w') as f:
    f.write(tex3)
print(f'Saved: {out3}')
print(tex3)

Saved: /Users/suzanne/Documents/GitHub/BayesCRNInference/writing/table3_comparison_l2.tex
% Table 3 — Bayesian vs LASSO polynomial L2 comparison
\begin{table}[htbp]
\centering
\small
\caption{%
  Polynomial-space $L^2$ error comparison:
  Bayesian spike-and-slab (\textbf{Bayes}) vs.\ sparse-learning-CRN
  (\textbf{LASSO}, fixed $\lambda = 0.01$).
  $L^2 = \|\hat{c} - c_{\mathrm{true}}\|_2$ in the 15-dimensional
  polynomial basis. ``(r)\rq\rq\/ = results still running.
}
\label{tab:comparison-l2}
\begin{tabular}{rlr|rr|rr|rr}
\toprule
\multicolumn{3}{c}{} & \multicolumn{2}{c}{$T=100$} & \multicolumn{2}{c}{$T=200$} & \multicolumn{2}{c}{$T=400$} \\
\cmidrule(lr){4-5}\cmidrule(lr){6-7}\cmidrule(lr){8-9}
\textbf{Ch} & \textbf{Change} & \textbf{Obs} & Bayes & LASSO & Bayes & LASSO & Bayes & LASSO \\
\midrule
0 & $-2A$ & 60 & \textbf{0.008} & 0.487 & \textit{(r)} & 0.374 & \textit{(r)} & 0.317 \\
1 & $-2A$, $+1X$, $+1Y$ & 129 & \textbf{0.060} & 0.754 & \textit{(r)} & 0.851 & \textit{(r)} & 0

In [17]:
# Save Table 3b — detailed Bayesian rate estimates (T=100 only for now)
def save_mcmc_detail_latex(T):
    if T not in mcmc_dfs:
        print(f'T={T} not available.')
        return
    df_m = mcmc_dfs[T]

    lines = []
    lines.append(rf'% Table 3b — Bayesian spike-and-slab rate estimates T={T}')
    lines.append(r'\begin{table}[htbp]')
    lines.append(r'\centering')
    lines.append(r'\small')
    lines.append(rf'\caption{{%')
    lines.append(rf'  Bayesian spike-and-slab posterior estimates for $T={T}$.')
    lines.append(r'  Columns: true rate $k$, posterior mean $\hat{k}$, posterior std, 95\% credible interval,')
    lines.append(r'  and probability of being \emph{off} (in the spike).')
    lines.append(r'  Only candidates with $\Pr(\text{off}) < 0.01$ or true $k > 0$ are shown.')
    lines.append(r'  Poly $L^2$ is computed in the 15-dim polynomial coefficient space.')
    lines.append(r'}')
    lines.append(rf'\label{{tab:bayes-detail-T{T}}}')
    lines.append(r'\begin{tabular}{rllrrrrrl}')
    lines.append(r'\toprule')
    lines.append(r'\textbf{Ch} & \textbf{Change} & \textbf{Reaction} & \textbf{True $k$} & $\hat{k}$ & Std & \textbf{95\% CI} & $\Pr(\text{off})$ & Poly $L^2$ \\')
    lines.append(r'\midrule')

    prev_ch = None
    for run_idx, grp in df_m.groupby('Run_Index'):
        cnt    = grp['Count'].iloc[0]
        change = count_to_change_ref.get(cnt)
        if change is None:
            continue
        key      = tuple(change)
        cands    = compatible_rxns.get(key, [])
        poly_l2  = mcmc_l2.get(T, {}).get(key, float('nan'))
        ch_i     = ref_changes.index(list(change))
        chg      = fmt_change_latex(change)

        first = True
        for _, r in grp.sort_values('Param_Index').iterrows():
            pi      = int(r['Param_Index'])
            true_k  = r['True_Theta']
            prob_off= r['Prob_Off']
            # Only show active params (true k>0 OR low Prob_Off)
            if true_k < 1e-6 and prob_off > 0.99:
                continue
            rxn_idx  = cands[pi] if pi < len(cands) else None
            rxn_name = name_by_idx.get(rxn_idx, '—') if rxn_idx else '—'
            rxn_lat  = fmt_rxn_latex(rxn_name) if rxn_name != '—' else '---'
            ci_str   = f"[{r['CI_lower']:.3f},\\,{r['CI_upper']:.3f}]"
            l2_str   = f'{poly_l2:.4f}' if (first and not np.isnan(poly_l2)) else ''
            ch_str   = str(ch_i) if first else ''
            chg_str  = chg if first else ''

            # Bold the true k if nonzero
            tk_str = (r'\textbf{' + f'{true_k:.4f}' + r'}') if true_k > 1e-6 else f'{true_k:.4f}'

            lines.append(
                f'{ch_str} & {chg_str} & {rxn_lat} & {tk_str} & '
                f"{r['Mean']:.4f} & {r['Std']:.4f} & "
                f'{ci_str} & {prob_off:.4f} & {l2_str} \\\\'
            )
            first = False
        if prev_ch != ch_i:
            lines.append(r'\midrule')
        prev_ch = ch_i

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')
    tex = '\n'.join(lines)
    out = os.path.join(WRITING_DIR, f'table3b_bayes_detail_T{T}.tex')
    with open(out, 'w') as f:
        f.write(tex)
    print(f'Saved: {out}')

for T in T_VALUES:
    save_mcmc_detail_latex(T)

Saved: /Users/suzanne/Documents/GitHub/BayesCRNInference/writing/table3b_bayes_detail_T100.tex
T=200 not available.
T=400 not available.


---
## Summary of saved LaTeX files

| File | Description |
|------|-------------|
| `table1_true_system.tex` | True 20-reaction CRN description |
| `table2a_lasso_fixed_lambda.tex` | LASSO L2 vs T at fixed λ |
| `table2b_lasso_lambda_sweep.tex` | LASSO L2 vs λ at T=200 |
| `table3_comparison_l2.tex` | Bayes vs LASSO polynomial L2 (all T) |
| `table3b_bayes_detail_T{N}.tex` | Detailed rate estimates + Prob(off) per T |

To include in Overleaf: `\\input{table1_true_system}` etc. (requires `booktabs`).